# Algorithm 5.2 — Lambert's problem

**Goal:** given two position vectors $\mathbf{R}_1,\mathbf{R}_2$ and the **time of flight** $\Delta t$ between them, find the orbit that connects them — i.e. the velocities $\mathbf{V}_1,\mathbf{V}_2$ at the two ends.

**Code (answer key):** [`lambert`](../curtis/algorithms/alg_5_2_lambert.py) · **Book:** §5.3, Algorithm 5.2 · **Worked example:** 5.2

This is the workhorse of mission design — transfers, rendezvous, porkchop plots all rest on it. It cashes in the **universal variable + Stumpff functions** from Chapter 3.

## Read first

| Symbol | Meaning |
|---|---|
| $\mathbf{R}_1,\mathbf{R}_2$ | start and end position vectors (km) |
| $\Delta t$ | time of flight from $\mathbf{R}_1$ to $\mathbf{R}_2$ (s) |
| `string` | `'pro'` (prograde) or `'retro'` (retrograde) — which way round |
| $\Delta\theta$ | swept true-anomaly angle between $\mathbf{R}_1$ and $\mathbf{R}_2$ |
| $A$ | geometry constant (Eq 5.35) |
| $z$ | universal variable solved for (sign = ellipse/parabola/hyperbola) |
| $\mathbf{V}_1,\mathbf{V}_2$ | the answer: velocities at the two ends |

**The core idea:** the time of flight is a single monotonic function of $z$; set it equal to the given $\Delta t$ and solve $F(z)=0$ for $z$.

## The picture (two points and a stopwatch)

Through any two points pass *infinitely many* orbits — fat ones, thin ones, fast and slow. Fixing the **time of flight** (and the direction, and the number of revolutions) selects exactly one. That is Lambert's problem, and solving it is how you plan a transfer: "I am here, I want to be there in $\Delta t$ — what burn do I need?"

The solution method:

1. **Geometry** — the two positions give the swept angle $\Delta\theta$ (and `'pro'`/`'retro'` picks the short or long way). This sets the constant $A$.
2. **One equation** — the time of flight depends on a single universal variable $z$ through the Stumpff functions $C(z), S(z)$. Solve $F(z)=\sqrt{\mu}\,\Delta t$ for $z$ with Newton's method.
3. **Velocities** — $z$ gives the Lagrange coefficients $f,g,\dot g$, and those give $\mathbf{V}_1,\mathbf{V}_2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math, cmath
import sys; sys.path.insert(0, "..")
from curtis.algorithms.alg_4_1_coe_from_sv import coe_from_sv            # read off elements
from curtis.algorithms.alg_5_2_lambert import lambert as _lambert_ref    # reference, for the picture

In [ ]:
# Example 5.2: connect R1 -> R2 in dt = 3600 s, and draw the transfer arc.
mu = 398600.0
R1 = np.array([5000.0, 10000.0, 2100.0])
R2 = np.array([-14600.0, 2500.0, 7000.0])
dt = 3600.0

V1, V2 = _lambert_ref(R1, R2, dt, "pro", mu)                 # reference solve (drawing only)
h, e, RA, incl, w, TA1, a = coe_from_sv(R1, V1, mu)
TA2 = coe_from_sv(R2, V2, mu)[5]

p_sl = h**2/mu
th = np.linspace(0, 2*np.pi, 500)
rr = p_sl/(1 + e*np.cos(th))
ox, oy = rr*np.cos(th), rr*np.sin(th)                        # full orbit (perifocal)

ta1, ta2 = TA1, TA2 + (2*np.pi if TA2 < TA1 else 0)          # prograde: sweep forward
arc = np.linspace(ta1, ta2, 200)
ra = p_sl/(1 + e*np.cos(arc))
ax_, ay = ra*np.cos(arc), ra*np.sin(arc)                     # the transfer arc
P1 = (p_sl/(1+e*np.cos(TA1)))*np.array([np.cos(TA1), np.sin(TA1)])
P2 = (p_sl/(1+e*np.cos(TA2)))*np.array([np.cos(TA2), np.sin(TA2)])

fig, ax = plt.subplots(figsize=(6.6, 6.2))
ax.plot(ox, oy, color='#c2c0b6', lw=1.2, ls='--')           # rest of the orbit
ax.plot(ax_, ay, color='#378ADD', lw=3)                      # the transfer arc
ax.plot(0, 0, 'o', color='#3B8BD4', ms=9)
ax.plot(*P1, 'o', color='#1D9E75', ms=7); ax.plot(*P2, 'o', color='#D4537E', ms=7)
ax.annotate('Earth', (0,0), textcoords='offset points', xytext=(8,-12), color='#3B8BD4')
ax.annotate(r'$R_1$ (start)', P1, textcoords='offset points', xytext=(8,4), color='#1D9E75')
ax.annotate(r'$R_2$ (after $\Delta t$)', P2, textcoords='offset points', xytext=(8,4), color='#D4537E')
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Lambert: the orbit through R1 and R2 with time of flight 3600 s')
plt.show()

## See it — the time of flight picks this arc

The blue arc is the orbit Lambert found: the one path from $\mathbf{R}_1$ to $\mathbf{R}_2$ that takes *exactly* 3600 s, prograde. The dashed remainder is the rest of that ellipse, which the spacecraft never flies.

Change the stopwatch and you change the arc: a longer $\Delta t$ fattens the ellipse (slower transfer), a much shorter one would force a hyperbola. Flip to `'retro'` and you would sweep the long way round instead. That one-orbit-per-(time, direction) selection is exactly what makes Lambert the basis of transfer and rendezvous design.

## Where these equations come from

### Geometry → the swept angle and $A$
The angle between the positions is $\Delta\theta=\cos^{-1}\!\big(\mathbf{R}_1\!\cdot\!\mathbf{R}_2/(r_1 r_2)\big)$, taken the short or long way according to the sign of $(\mathbf{R}_1\times\mathbf{R}_2)_z$ versus `'pro'`/`'retro'`. From it,
$$A=\sin\Delta\theta\,\sqrt{\frac{r_1 r_2}{1-\cos\Delta\theta}}\qquad(5.35),$$
a constant fixed entirely by the geometry.

### One equation in one unknown
The universal variable $z$ (same $z=\alpha\chi^2$ as Chapter 3) parametrises the family of connecting orbits. Define
$$y(z)=r_1+r_2+A\,\frac{z\,S(z)-1}{\sqrt{C(z)}}\qquad(5.38),$$
then the time of flight is
$$\sqrt{\mu}\,\Delta t=\Big(\tfrac{y(z)}{C(z)}\Big)^{3/2}S(z)+A\sqrt{y(z)}\qquad(5.40).$$
Moving $\sqrt\mu\,\Delta t$ across gives $F(z)=0$ — **one equation, one unknown** — solved by Newton's method with derivative $F'(z)$ (Eq 5.43). The sign of the root $z$ reports the conic (ellipse $z>0$, hyperbola $z<0$).

### $z$ → the velocities
With $z$ (hence $y$) known, the Lagrange coefficients are $f=1-\tfrac{y}{r_1}$, $g=A\sqrt{\tfrac{y}{\mu}}$, $\dot g=1-\tfrac{y}{r_2}$ (Eqs 5.46), and then
$$\mathbf{V}_1=\frac{1}{g}(\mathbf{R}_2-f\mathbf{R}_1),\qquad \mathbf{V}_2=\frac{1}{g}(\dot g\,\mathbf{R}_2-\mathbf{R}_1)\qquad(5.28,\,5.29).$$

### A real implementation gotcha (why the helpers use `cmath`)
Newton needs a starting bracket, found by sweeping $z$ up from $-100$ until $F$ changes sign. In that low-$z$ region $y(z)$ can go **negative**, so $\sqrt{y}$ and $(y/C)^{3/2}$ are complex. The clean fix is to compute $y,F,F'$ in **complex arithmetic** and compare/extract the real part — the converged $z$ is real and physical. The helper cell below does exactly this, so you can focus on the main flow.

In [ ]:
# --- Provided machinery (formula transcription + the cmath guard). ---
# Complex-aware Stumpff functions; identical to stumpC/stumpS on the real axis.
def _C(z):
    if z == 0: return 0.5
    return (1 - cmath.cos(cmath.sqrt(z)))/z

def _S(z):
    if z == 0: return 1/6
    sz = cmath.sqrt(z)
    return (sz - cmath.sin(sz))/sz**3

def _y(z, A, r1, r2):                    # Eq 5.38
    return r1 + r2 + A*(z*_S(z) - 1)/cmath.sqrt(_C(z))

def _F(z, t, A, r1, r2, mu):             # Eq 5.40, moved to one side
    y = _y(z, A, r1, r2)
    return (y/_C(z))**1.5 * _S(z) + A*cmath.sqrt(y) - math.sqrt(mu)*t

def _dFdz(z, A, r1, r2):                 # Eq 5.43
    if z == 0:
        y0 = _y(0, A, r1, r2)
        return math.sqrt(2)/40*y0**1.5 + A/8*(cmath.sqrt(y0) + A*cmath.sqrt(1/2/y0))
    C, S, y = _C(z), _S(z), _y(z, A, r1, r2)
    return ((y/C)**1.5 * (1/(2*z)*(C - 3*S/(2*C)) + 3*S**2/(4*C))
            + A/8*(3*S/C*cmath.sqrt(y) + A*cmath.sqrt(C/y)))

## Step by step (in code order)

1. **Magnitudes & swept angle:** $r_1,r_2$; $\;\Delta\theta=\cos^{-1}(\mathbf{R}_1\!\cdot\!\mathbf{R}_2/(r_1 r_2))$; then flip to the long way if the direction calls for it (sign of $(\mathbf{R}_1\times\mathbf{R}_2)_z$).
2. **Geometry constant:** $A=\sin\Delta\theta\sqrt{r_1 r_2/(1-\cos\Delta\theta)}$.
3. **Bracket the root:** start $z=-100$ and step up by $0.1$ while $\mathrm{Re}\,F(z)<0$.
4. **Newton:** `ratio = Re[F/F']`; `z -= ratio`; repeat until `|ratio| < tol`.
5. **Lagrange coefficients** from $y=\mathrm{Re}\,y(z)$: $f=1-y/r_1$, $g=A\sqrt{y/\mu}$, $\dot g=1-y/r_2$.
6. **Velocities:** $\mathbf{V}_1=(\mathbf{R}_2-f\mathbf{R}_1)/g$, $\;\mathbf{V}_2=(\dot g\,\mathbf{R}_2-\mathbf{R}_1)/g$.

**↓ Now type the main flow below**, using the provided `_F`, `_dFdz`, `_y`. The helpers return complex numbers, so take `.real` where the comments say so. Answer key linked above; peek only after you try.

In [ ]:
def lambert(R1, R2, t, string="pro", mu=398600.0, tol=1.0e-8, nmax=5000):
    """Velocities V1, V2 of the orbit through R1, R2 with time of flight t."""
    R1 = np.asarray(R1, float); R2 = np.asarray(R2, float)
    r1 = np.linalg.norm(R1); r2 = np.linalg.norm(R2)
    c12 = np.cross(R1, R2)
    theta = math.acos(np.dot(R1, R2)/(r1*r2))

    # 1. Prograde/retrograde: flip theta to the long way if needed
    #        if string == 'pro'  and c12[2] <= 0:  theta = 2*pi - theta
    #        if string == 'retro' and c12[2] >= 0:  theta = 2*pi - theta

    # 2. Geometry constant (Eq 5.35):
    #        A = sin(theta) * sqrt(r1*r2 / (1 - cos(theta)))

    # 3. Bracket the root:
    #        z = -100
    #        while _F(z, t, A, r1, r2, mu).real < 0:  z += 0.1

    # 4. Newton's method:
    #        ratio = 1
    #        while abs(ratio) > tol:
    #            ratio = (_F(z, t, A, r1, r2, mu) / _dFdz(z, A, r1, r2)).real
    #            z = z - ratio

    # 5. Lagrange coefficients (Eqs 5.46), using y = _y(z, A, r1, r2).real :
    #        f    = 1 - y/r1
    #        g    = A*sqrt(y/mu)
    #        gdot = 1 - y/r2

    # 6. Velocities (Eqs 5.28, 5.29):
    #        V1 = (R2 - f*R1)/g
    #        V2 = (gdot*R2 - R1)/g

    # return V1, V2
    raise NotImplementedError("fill in steps 1-6, then delete this line")

## Verify — Example 5.2

**Input** ($\mu_\oplus=398600$): $\mathbf{R}_1=[5000,10000,2100]$, $\mathbf{R}_2=[-14600,2500,7000]$ km, $\Delta t=3600$ s, prograde.

**Expected:** $\mathbf{V}_1=[-5.99249, 1.92536, 3.24564]$, $\mathbf{V}_2=[-3.31246, -4.19662, -0.385288]$ km/s; orbit $e=0.433488$, $i=30.191°$, $a=20002.9$ km.

Run once your function is typed.

In [ ]:
mu = 398600.0
R1 = np.array([5000.0, 10000.0, 2100.0])
R2 = np.array([-14600.0, 2500.0, 7000.0])
dt = 3600.0

V1, V2 = lambert(R1, R2, dt, "pro", mu)
print(f"V1 (km/s) = [{V1[0]:.6g} {V1[1]:.6g} {V1[2]:.6g}]   (expect [-5.99249  1.92536  3.24564])")
print(f"V2 (km/s) = [{V2[0]:.6g} {V2[1]:.6g} {V2[2]:.6g}]   (expect [-3.31246  -4.19662  -0.385288])")

assert np.allclose(V1, [-5.99249, 1.92536, 3.24564], atol=1e-4)
assert np.allclose(V2, [-3.31246, -4.19662, -0.385288], atol=1e-4)

deg = np.pi/180
h, e, RA, incl, w, TA, a = coe_from_sv(R1, V1, mu)
print(f"\ne = {e:.6g}  (0.433488)   i = {incl/deg:.5g} deg  (30.191)   a = {a:.6g} km  (20002.9)")
assert abs(e - 0.433488) < 1e-5
assert abs(incl/deg - 30.191) < 1e-2
assert abs(a - 20002.9) < 1e-1
print("\nAll checks passed ✔")

## What this confirms

- **Two positions plus a clock determine an orbit.** Fixing $\Delta t$ (and direction) collapses the infinite family of connecting orbits to one — solved as a single root $F(z)=0$.
- The Chapter-3 **universal variable and Stumpff functions** carry straight over; Lambert is "Kepler's problem run the other way."
- It outputs the full state at both ends, so chaining with `coe_from_sv` gives the transfer orbit's elements — the basis of a delta-v / transfer calculation.

**Next:** Algorithm 5.3 (`LST`) — local sidereal time, the timekeeping needed to turn ground-based observations into the geocentric position vectors that Gibbs and the angles-only methods consume.